# RetailIQ 360° — ETL de Integración

**Objetivo:** Resolver los tres problemas estructurales del modelo de datos antes de construir medidas DAX.

---

## Diagnóstico de partida
| Problema | Impacto en Power BI |
|---|---|
| `fact_ventas` sin contexto argentino (sin CanalID, medio_pago, etc.) | No se puede cruzar precio real con canal de venta |
| `fact_ventas_base` sin precios (100% nulos) | No se puede calcular facturación por canal |
| `DimClientesAr` sin `GeografiaID` | No se puede filtrar clientes por provincia en el mapa |
| `DimInflacionIpc` sin clave relacional (`PeriodoID`) | La relación con FactVentas no es estándar en Power BI |

## Solución: tres tareas

1. **`fact_ventas_final.csv`** — enriquecer `fact_ventas` (precios reales de Olist) con contexto argentino sintético calibrado con CACE 2025
2. **`dim_clientes_ar.csv`** — agregar `GeografiaID` a DimClientesAr via merge con DimGeografia
3. **`dim_inflacion_ipc.csv`** — agregar `PeriodoID = anio * 100 + mes` para relación real en Power BI

**Inputs:** `datos/03_sinteticos/` y `datos/04_procesados/`  
**Outputs:** `datos/04_procesados/` (tres archivos nuevos/actualizados)

## BLOQUE 0 — Configuración y carga

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

SINT_DIR = '../datos/03_sinteticos'
PROC_DIR = '../datos/04_procesados'
SEED     = 42
rng      = np.random.default_rng(SEED)

# ── Cargar todos los inputs ──────────────────────────────────
fact_ventas   = pd.read_csv(f'{PROC_DIR}/fact_ventas.csv', parse_dates=['fecha'])
dim_clientes  = pd.read_csv(f'{SINT_DIR}/004_dim_clientes_ar.csv')
dim_geografia = pd.read_csv(f'{SINT_DIR}/001_dim_geografia_ar.csv')
dim_canal     = pd.read_csv(f'{SINT_DIR}/003_dim_canal_ar.csv')
dim_sucursal  = pd.read_csv(f'{SINT_DIR}/002_dim_sucursales_ar.csv')
dim_inflacion = pd.read_csv(f'{PROC_DIR}/dim_inflacion_ipc.csv')

print('✓ Archivos cargados')
print(f'  fact_ventas:   {len(fact_ventas):,} filas | cols: {list(fact_ventas.columns)}')
print(f'  dim_clientes:  {len(dim_clientes):,} filas')
print(f'  dim_geografia: {len(dim_geografia):,} filas')
print(f'  dim_sucursal:  {len(dim_sucursal):,} filas')
print(f'  dim_inflacion: {len(dim_inflacion):,} filas')

print()
print('DimCanal (referencia para pesos):')
print(dim_canal[['CanalID', 'nombre', 'peso_facturacion']].to_string(index=False))

---
## BLOQUE 1 — `fact_ventas_final`: enriquecer Olist con contexto argentino

La estrategia es tomar `fact_ventas` como base (tiene precios reales en ARS, deflactados por IPC)
y agregarle las columnas de contexto argentino que tenía `fact_ventas_base` (canal, cliente, medio de pago, etc.).

Los valores sintéticos se samplearán usando las distribuciones reales de CACE 2025 para que
el dashboard refleje el mercado e-commerce argentino.

In [ ]:
n = len(fact_ventas)
base = fact_ventas.copy()

# ── CanalID: distribución según peso_facturacion CACE 2025 ──
# DimCanal: 1=Tienda física 8%, 2=Web 25%, 3=App 20%, 4=Marketplace 47%
base['CanalID'] = rng.choice([1, 2, 3, 4], size=n, p=[0.08, 0.25, 0.20, 0.47])

# ── ClienteID: sorteo aleatorio del padrón de DimClientesAr ─
# Cada transacción se asigna a un cliente existente. El mismo cliente
# puede tener múltiples compras, lo cual es realista (comprador recurrente).
clientes_ids = dim_clientes['ClienteID'].values
base['ClienteID'] = rng.choice(clientes_ids, size=n)

# ── SucursalID: solo para CanalID=1 (venta en tienda física) ─
# Las compras online (canal 2, 3, 4) no tienen sucursal física asignada.
# El NaN en Power BI aparece como "en blanco" en la relación — correcto.
sucursal_ids = dim_sucursal['SucursalID'].values
mask_fisica  = base['CanalID'] == 1
base['SucursalID'] = np.nan
base.loc[mask_fisica, 'SucursalID'] = rng.choice(
    sucursal_ids, size=int(mask_fisica.sum())
).astype(float)

# ── medio_pago: distribución CACE 04a (MID 2023) ────────────
# tarjeta_credito: ~74% (plataformas 58% + gateway 16%)
# debito:          ~9%
# transferencia:   ~5%
# efectivo:        ~12%  (ajustado para sumar 100%)
medios  = ['tarjeta_credito', 'debito', 'transferencia', 'efectivo']
p_medio = [0.74, 0.09, 0.05, 0.12]
base['medio_pago'] = rng.choice(medios, size=n, p=p_medio)

# ── nro_cuotas: depende del medio de pago ────────────────────
# Crédito: puede tener 1, 3, 6, 10 o 15 cuotas (CACE 04b 2025)
# Débito, transferencia, efectivo: siempre 1 cuota
cuotas_vals  = [1, 3, 6, 10, 15]
p_cuotas     = [0.27, 0.28, 0.26, 0.14, 0.05]
mask_credito = base['medio_pago'] == 'tarjeta_credito'

nro_cuotas = np.ones(n, dtype=int)
nro_cuotas[mask_credito] = rng.choice(
    cuotas_vals, size=int(mask_credito.sum()), p=p_cuotas
)
base['nro_cuotas'] = nro_cuotas

# ── tipo_entrega: distribución CACE 05a (2025) ───────────────
tipos_entrega = ['domicilio', 'retiro_local', 'retiro_operador', 'pickup', 'last_miler']
p_entrega     = [0.60, 0.29, 0.08, 0.02, 0.01]
base['tipo_entrega'] = rng.choice(tipos_entrega, size=n, p=p_entrega)

# ── plazo_entrega: distribución CACE 05b (2024) ──────────────
plazos   = ['same_day', '24hs', '48hs', 'semana', '15dias', 'mes_mas']
p_plazos = [0.17, 0.16, 0.14, 0.39, 0.11, 0.03]
base['plazo_entrega'] = rng.choice(plazos, size=n, p=p_plazos)

# ── cantidad: Olist tiene granularidad 1 ítem = 1 fila ────────
base['cantidad'] = 1

# ── PeriodoID: clave para relación con DimInflacionIpc ────────
# Formato yyyymm (ej: 201701 para enero 2017).
# Permite crear una relación estándar entre FactVentasFinal y DimInflacionIpc en Power BI.
base['PeriodoID'] = base['anio'] * 100 + base['mes']

print('✓ Columnas argentinas agregadas')
print()
print('CanalID (distribución real):')
canal_dist = base['CanalID'].value_counts().sort_index()
for cid, cnt in canal_dist.items():
    nombre = dim_canal.loc[dim_canal['CanalID']==cid, 'nombre'].values[0]
    print(f'  {cid} {nombre:<15}  {cnt:>7,}  ({cnt/n*100:.1f}%)')

print()
print('medio_pago (distribución real):')
for mp, cnt in base['medio_pago'].value_counts().items():
    print(f'  {mp:<20} {cnt:>7,}  ({cnt/n*100:.1f}%)')

print()
print(f'SucursalID asignado:  {mask_fisica.sum():,} filas (CanalID=1)')
print(f'SucursalID = NaN:     {(~mask_fisica).sum():,} filas (canales online)')

In [ ]:
# ── Renombrar columnas de precio para consistencia con el proyecto ──
# Se mantienen los tres precios para flexibilidad en el dashboard:
#   precio_venta_brl      → precio original Olist (referencia Brasil)
#   precio_venta_ars      → precio nominal ARS (el de la factura del período)
#   precio_venta_ars_real → precio deflactado a dic-2016 (para comparar períodos)
base = base.rename(columns={
    'price_brl':     'precio_venta_brl',
    'freight_brl':   'flete_brl',
    'price_ars':     'precio_venta_ars',
    'freight_ars':   'flete_ars',
    'price_ars_real':'precio_venta_ars_real',
})

# ── VentaID: clave surrogate secuencial (como la tenía FactVentasBase) ─
base.insert(0, 'VentaID', range(1, n + 1))

# ── Seleccionar y ordenar columnas finales ────────────────────
# Orden pensado para Power BI: claves → temporal → contexto → precios → técnico
cols_final = [
    # Claves dimensionales
    'VentaID', 'ClienteID', 'CanalID', 'SucursalID', 'PeriodoID',
    # Temporal
    'fecha', 'anio', 'mes', 'trimestre',
    # Producto
    'category_en',
    # Contexto argentino
    'medio_pago', 'nro_cuotas', 'tipo_entrega', 'plazo_entrega', 'cantidad',
    # Precios y conversión
    'precio_venta_brl', 'flete_brl',
    'ars_por_usd', 'tipo_cambio',
    'precio_venta_ars', 'flete_ars',
    # Inflación
    'ipc_nivel_general', 'indice_ipc_acum', 'precio_venta_ars_real', 'tiene_ipc',
    # IDs técnicos Olist (para trazabilidad con el dataset original)
    'order_id', 'product_id', 'seller_id',
]

fact_ventas_final = base[cols_final].copy()

print(f'fact_ventas_final lista')
print(f'  Filas:    {len(fact_ventas_final):,}')
print(f'  Columnas: {len(fact_ventas_final.columns)}')
print()
print('Muestra (columnas clave):')
print(fact_ventas_final[
    ['VentaID','ClienteID','CanalID','SucursalID','medio_pago',
     'precio_venta_ars','precio_venta_ars_real','PeriodoID']
].head(5).to_string(index=False))

---
## BLOQUE 2 — `DimClientesAr`: agregar `GeografiaID`

Actualmente `DimClientesAr` tiene `provincia` y `ciudad` como texto libre pero no tiene FK a `DimGeografia`.  
Esto impide relacionar clientes con la jerarquía geográfica en Power BI.

**Estrategia de join:**
1. Intentar match exacto por `(provincia, ciudad)` — da el GeografiaID más específico
2. Para los que no matcheen, usar el primer GeografiaID de esa `provincia`

In [ ]:
# ── Paso 1: tabla de lookup por (provincia, ciudad) ───────────
# Tomamos el primer GeografiaID por combinación provincia+ciudad.
# Si una ciudad tiene varias filas en DimGeografia (ej: CABA con barrios),
# tomamos la primera — es suficiente para el FK.
geo_ciudad = (
    dim_geografia
    .groupby(['provincia', 'ciudad'], as_index=False)['GeografiaID']
    .first()
)

# ── Paso 2: tabla de lookup por provincia (fallback) ──────────
geo_prov = (
    dim_geografia
    .groupby('provincia', as_index=False)['GeografiaID']
    .first()
    .rename(columns={'GeografiaID': 'GeografiaID_prov'})
)

# ── Paso 3: merge cascado ─────────────────────────────────────
dim_clientes_v2 = (
    dim_clientes
    .merge(geo_ciudad, on=['provincia', 'ciudad'], how='left')
    .merge(geo_prov,   on='provincia',             how='left')
)

# Usar match ciudad si existe, sino match provincia
dim_clientes_v2['GeografiaID'] = (
    dim_clientes_v2['GeografiaID']
    .fillna(dim_clientes_v2['GeografiaID_prov'])
    .astype('Int64')  # Int64 admite NaN en pandas (entero nullable)
)
dim_clientes_v2 = dim_clientes_v2.drop(columns=['GeografiaID_prov'])

# ── Verificación ──────────────────────────────────────────────
total      = len(dim_clientes_v2)
con_geo    = dim_clientes_v2['GeografiaID'].notna().sum()
sin_geo    = dim_clientes_v2['GeografiaID'].isna().sum()

print('DimClientesAr + GeografiaID')
print(f'  Total clientes:    {total:,}')
print(f'  Con GeografiaID:   {con_geo:,}  ({con_geo/total*100:.1f}%)')
print(f'  Sin GeografiaID:   {sin_geo:,}')

if sin_geo > 0:
    provincias_sin_match = (
        dim_clientes_v2[dim_clientes_v2['GeografiaID'].isna()]['provincia']
        .value_counts()
    )
    print(f'\n  Provincias sin match en DimGeografia:')
    print(provincias_sin_match.to_string())

print()
print('Muestra con GeografiaID:')
print(dim_clientes_v2[['ClienteID','provincia','ciudad','GeografiaID']].head(5).to_string(index=False))

---
## BLOQUE 3 — `DimInflacionIpc`: agregar `PeriodoID`

La relación entre `DimInflacionIpc` y `FactVentasFinal` en Power BI necesita una columna clave compartida.
Ambas tablas ya tienen `anio` y `mes` como enteros. Creamos `PeriodoID = anio * 100 + mes`
(ej: enero 2017 → 201701) que actúa como clave única de esa relación.

In [ ]:
# PeriodoID en DimInflacionIpc
dim_inflacion['PeriodoID'] = dim_inflacion['anio'] * 100 + dim_inflacion['mes']

print('DimInflacionIpc + PeriodoID')
print(f'  Filas: {len(dim_inflacion):,}')
print(f'  PeriodoID cubre: {dim_inflacion["PeriodoID"].min()} → {dim_inflacion["PeriodoID"].max()}')
print()
print('Muestra:')
print(dim_inflacion[['fecha','anio','mes','PeriodoID','ipc_nivel_general']].head(6).to_string(index=False))

# Verificar que fact_ventas_final.PeriodoID está cubierto por dim_inflacion.PeriodoID
periodos_fact = set(fact_ventas_final['PeriodoID'].unique())
periodos_ipc  = set(dim_inflacion['PeriodoID'].unique())
sin_cobertura = periodos_fact - periodos_ipc

print()
if sin_cobertura:
    print(f'  Períodos en FactVentasFinal sin IPC: {sorted(sin_cobertura)}')
    print('  (son las filas con tiene_ipc=False — sep-dic 2016, ya manejadas en ETL anterior)')
else:
    print('  ✓ Todos los PeriodoID de FactVentasFinal tienen IPC en DimInflacionIpc')

---
## BLOQUE 4 — Guardado y resumen final

In [ ]:
os.makedirs(PROC_DIR, exist_ok=True)

# Guardar los tres archivos en datos/04_procesados/
ruta_fact    = f'{PROC_DIR}/fact_ventas_final.csv'
ruta_cli     = f'{PROC_DIR}/dim_clientes_ar.csv'
ruta_ipc     = f'{PROC_DIR}/dim_inflacion_ipc.csv'

fact_ventas_final.to_csv(ruta_fact, index=False, encoding='utf-8-sig')
dim_clientes_v2.to_csv(ruta_cli,   index=False, encoding='utf-8-sig')
dim_inflacion.to_csv(ruta_ipc,     index=False, encoding='utf-8-sig')

print('=' * 65)
print('RESUMEN DE ARCHIVOS GENERADOS')
print('=' * 65)
print()
print(f'[1] fact_ventas_final.csv  →  datos/04_procesados/')
print(f'    Filas:    {len(fact_ventas_final):,}')
print(f'    Columnas: {len(fact_ventas_final.columns)}')
print(f'    Período:  {fact_ventas_final["fecha"].min().date()} → {fact_ventas_final["fecha"].max().date()}')
print(f'    Facturación total ARS: ${fact_ventas_final["precio_venta_ars"].sum():,.0f}')
print()
print(f'[2] dim_clientes_ar.csv    →  datos/04_procesados/')
print(f'    Filas:    {len(dim_clientes_v2):,}')
print(f'    Columnas: {len(dim_clientes_v2.columns)}  (GeografiaID agregado)')
print(f'    Clientes con GeografiaID: {dim_clientes_v2["GeografiaID"].notna().sum():,}')
print()
print(f'[3] dim_inflacion_ipc.csv  →  datos/04_procesados/')
print(f'    Filas:    {len(dim_inflacion):,}')
print(f'    Columnas: {len(dim_inflacion.columns)}  (PeriodoID agregado)')
print()
print('=' * 65)
print('PROXIMOS PASOS EN POWER BI')
print('=' * 65)
print()
print('1. Actualizar origen de datos:')
print('   FactVentas     → fact_ventas_final.csv  (04_procesados)')
print('   DimClientesAr  → dim_clientes_ar.csv    (04_procesados)')
print('   DimInflacionIpc→ dim_inflacion_ipc.csv  (04_procesados) [mismo path]')
print()
print('2. Crear relaciones en Vista de Modelo:')
print('   FactVentasFinal[CanalID]      → DimCanal[CanalID]')
print('   FactVentasFinal[ClienteID]    → DimClientesAr[ClienteID]')
print('   FactVentasFinal[SucursalID]   → DimSucursalesAr[SucursalID]')
print('   FactVentasFinal[PeriodoID]    → DimInflacionIpc[PeriodoID]')
print('   DimClientesAr[GeografiaID]    → DimGeografia[GeografiaID]')
print('   DimSucursalesAr[GeografiaID]  → DimGeografia[GeografiaID]')
print()
print('3. Reactivar DimTiempo como tabla de fechas con fecha')
print('   y relacionarla con FactVentasFinal[fecha]')